<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose

This notebook converts the laboratory specification into a **complete end-to-end algorithmic workflow** before implementation. It is intentionally positioned between the theory/requirements notebooks and `main.ipynb`.

Use it to answer four questions for every stage:

1. **What enters this stage?**
2. **What operation or decision is performed?**
3. **What must leave the stage?**
4. **What must be checked before moving forward?**

The numbered stages below match the Problem Statement and `main.ipynb` exactly.


## End-to-End Flow

```text
images → refined corners → planar correspondences → normalized homographies → Zhang constraints → intrinsics → poses → reprojection → validation
```


## Inputs and Final Outputs

| | Description |
| --- | --- |
| **Inputs** | chessboard images, 8×6 internal corners, 0.03 m square size |
| **Final outputs** | intrinsic matrix K, one pose (R,t) per valid view, reprojection metrics, six diagnostic figures |
| **Execution rule** | Every stage consumes only validated outputs from earlier stages. |
| **Failure rule** | Stop on missing data, invalid dimensions, non-finite values, degenerate geometry, or missing declared artifacts. |


## 1. Load the Sorted JPEG Calibration Images

**Procedure**

1. Discover the required repository-relative input files and the lab output directory.
2. Read the selected inputs in the representation required by the workflow.
3. Reject missing, unreadable, or unsupported data before any downstream computation.

**Output.** Validated input data and output paths.

**Acceptance check.** Every required input exists, is readable, and has the expected basic dimensions.

**Handoff.** Pass only this validated result to Stage 2; the implementation must not silently replace it with an unchecked alternative.


## 2. Detect and Refine Chessboard Corners

**Procedure**

1. Convert each calibration view to grayscale and detect the complete 8×6 chessboard pattern.
2. Refine the detected corners to sub-pixel precision.
3. Retain only views containing all 48 internal corners.

**Output.** Refined 2-D image correspondences for every valid view.

**Acceptance check.** Each retained corner array is finite and has shape (48, 2).

**Handoff.** Pass only this validated result to Stage 3; the implementation must not silently replace it with an unchecked alternative.


## 3. Build the Planar World Coordinates

**Procedure**

1. Generate the 8×6 chessboard grid in the same ordering as the detected corners.
2. Scale x/y coordinates by the 0.03 m physical square size.
3. Represent the calibration board on the plane Z=0.

**Output.** Metric planar calibration coordinates.

**Acceptance check.** Point count and ordering match the image correspondences.

**Handoff.** Pass only this validated result to Stage 4; the implementation must not silently replace it with an unchecked alternative.


## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

**Procedure**

1. Use only validated outputs from the preceding stage.
2. Execute the operation defined by this stage with the same representation and parameters used in main.ipynb.
3. Store the result under the semantic role expected by the next stage.

**Output.** A validated intermediate result for this stage.

**Acceptance check.** Confirm expected shape, range, finiteness, and logical consistency before handoff.

**Handoff.** Pass only this validated result to Stage 5; the implementation must not silently replace it with an unchecked alternative.


## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

**Procedure**

1. Build two DLT equations for every normalized plane-to-image correspondence.
2. Stack the equations into Q and solve Qh=0 with SVD.
3. Reshape the last right-singular vector into the normalized 3×3 homography.

**Output.** One normalized homography per retained view.

**Acceptance check.** The solution is finite and Q has enough independent constraints.

**Handoff.** Pass only this validated result to Stage 6; the implementation must not silently replace it with an unchecked alternative.


## 6. Denormalize Each Homography

**Procedure**

1. Use only validated outputs from the preceding stage.
2. Execute the operation defined by this stage with the same representation and parameters used in main.ipynb.
3. Store the result under the semantic role expected by the next stage.

**Output.** A validated intermediate result for this stage.

**Acceptance check.** Confirm expected shape, range, finiteness, and logical consistency before handoff.

**Handoff.** Pass only this validated result to Stage 7; the implementation must not silently replace it with an unchecked alternative.


## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

**Procedure**

1. Extract the first two columns of every homography.
2. Build the orthogonality and equal-norm Zhang constraints.
3. Stack them into V and solve Vb=0 with SVD.

**Output.** The scale-ambiguous vector b defining the intrinsic constraints.

**Acceptance check.** V has the expected multi-view shape and the residual norm is finite.

**Handoff.** Pass only this validated result to Stage 8; the implementation must not silently replace it with an unchecked alternative.


## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

**Procedure**

1. Convert intensities to a numerically safe representation when arithmetic is required.
2. Apply the stated pointwise mapping with explicit parameters.
3. Clip/cast to the intended display range and compare the response.

**Output.** Intensity-transformed image and, where relevant, response curve.

**Acceptance check.** uint8 results stay in [0,255] and identity cases behave as expected.

**Handoff.** Pass only this validated result to Stage 9; the implementation must not silently replace it with an unchecked alternative.


## 9. Recover $R$ and $t$ for Every Retained View

**Procedure**

1. Back-project homography columns with K inverse and recover the common scale.
2. Form r1, r2, r3 and translation t.
3. Project the approximate rotation to the nearest proper orthonormal rotation with SVD.

**Output.** One physically valid pose (R,t) per view.

**Acceptance check.** R transpose R is approximately identity and det(R) is approximately +1.

**Handoff.** Pass only this validated result to Stage 10; the implementation must not silently replace it with an unchecked alternative.


## 10. Reproject the $Z=0$ Calibration Points

**Procedure**

1. Lift each calibration point to (X,Y,0) and transform it into the camera frame.
2. Project through K and divide by homogeneous depth.
3. Pair predicted image points with their measured refined corners.

**Output.** Reprojected image coordinates for all retained points.

**Acceptance check.** Homogeneous depth is non-zero and all projected coordinates are finite.

**Handoff.** Pass only this validated result to Stage 11; the implementation must not silently replace it with an unchecked alternative.


## 11. Compute Point-wise Errors, Mean Error and RMSE

**Procedure**

1. Apply the requested smoothing or denoising operator with explicit parameters.
2. Keep source image and boundary handling fixed across the comparison.
3. Measure both noise suppression and detail preservation where reference data allow.

**Output.** Filtered image for the selected denoising method.

**Acceptance check.** Output shape matches input and the comparison changes only the intended method/parameter.

**Handoff.** Pass only this validated result to Stage 12; the implementation must not silently replace it with an unchecked alternative.


## 12. Produce and Save the Six Required Diagnostic Figures

**Procedure**

1. Render the current result using the same data produced by the computation stage.
2. Annotate the diagnostic quantity that makes the stage interpretable.
3. Save the figure under its declared repository output name.

**Output.** Reproducible diagnostic figure.

**Acceptance check.** Saved artifact exists and corresponds to the displayed computation.

**Handoff.** Pass only this validated result to Stage 13; the implementation must not silently replace it with an unchecked alternative.


## 13. Run the Numerical and Output-file Validation Checks

**Procedure**

1. Check the accumulated arrays, matrices, metrics, and structures against explicit invariants.
2. Reject non-finite, out-of-range, dimensionally invalid, or logically inconsistent results.
3. Verify every filename declared by REQUIRED_OUTPUTS exists.

**Output.** Final pass/fail validation state.

**Acceptance check.** The lab completes only when every assertion and output-file check passes.

**Handoff.** Pass this validated result to the final workflow result; the implementation must not silently replace it with an unchecked alternative.


## Complete End-to-End Pseudocode

```text
DISCOVER and sort calibration images
FOR each image:
    READ image and convert to grayscale
    DETECT 8×6 chessboard corners
    IF detection is complete:
        REFINE corners to sub-pixel precision
        RETAIN the view
REQUIRE at least three retained views
BUILD planar metric chessboard coordinates on Z = 0

FOR each retained view:
    NORMALIZE plane points and image points
    BUILD normalized DLT system Q
    SOLVE Qh = 0 with SVD
    RESHAPE h into normalized homography
    DENORMALIZE and scale H

STACK Zhang constraints from every homography into V
SOLVE Vb = 0 with SVD
RECOVER alpha, beta, gamma, u0, v0
CONSTRUCT K and reject non-physical solutions

FOR each homography:
    RECOVER r1, r2 and t from K^-1 H
    COMPUTE r3 = r1 × r2
    PROJECT approximate rotation onto SO(3)
    STORE R and t

FOR each retained view:
    REPROJECT all Z = 0 calibration points
    COMPUTE point-wise pixel residuals, mean error and RMSE

GENERATE the six required diagnostic figures
VALIDATE matrix shapes, finiteness, rotation constraints, residuals and output files
RETURN K, poses, metrics and figures
```

This pseudocode is the compact execution contract for the lab. The detailed mathematical justification remains in `theory.ipynb`, environment/data preparation remains in `requirements.ipynb`, and executable code/results remain in `main.ipynb`.


## Mapping to the Lab Notebooks

| Notebook | Role |
| --- | --- |
| `problem_statement.ipynb` | Defines what must be solved and delivered. |
| `theory.ipynb` | Explains the mathematical and conceptual basis. |
| `requirements.ipynb` | Defines software, data, and execution prerequisites. |
| `algorithm.ipynb` | Defines the complete ordered procedure, decisions, handoffs, and validation logic. |
| `main.ipynb` | Executes the algorithm and produces numerical/visual results. |

The algorithm notebook should therefore be read **before** `main.ipynb` when learning or reviewing the lab.
